# Psoriasis scRNA-seq Drug Repurposing Pipeline
**Transcriptome-Guided Drug Repurposing for Psoriasis**

Glen Ritschel | Ritschel Research | 2026

GitHub: https://github.com/glenritschel/psoriasis-scrna

**Dataset**: GSE173706 (Ma et al. 2023)
6 patients x 2 conditions: PP (lesional) + PN (uninvolved)
12 biopsies total, MTX format

**Key advantage**: Paired PP vs PN design enables true disease-specific DE

**Before running:** Runtime > Change runtime type > T4 GPU

## Step 1: Clone repo and install dependencies

In [ ]:
import os
REPO_URL = "https://github.com/glenritschel/psoriasis-scrna"
REPO_DIR = "psoriasis-scrna"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}/notebooks
print("Working directory:", os.getcwd())


In [ ]:
import subprocess, sys
packages = ["scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10",
            "leidenalg", "python-igraph", "scikit-misc", "biopython"]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")


## Step 2: Download GSE173706 from NCBI GEO

GSE173706 uses MTX format (barcodes/features/matrix) not h5.
Each GSM is a separate directory with 3 files.

In [ ]:
import os, urllib.request, tarfile, gzip, shutil

RAW_DIR = "/content/psoriasis-scrna/data/raw/GSE173706"
os.makedirs(RAW_DIR, exist_ok=True)

# GSE173706: 12 samples (6 PP lesional + 6 PN uninvolved)
SAMPLES = [
    ("GSM5284973", "PP", "P1"), ("GSM5284974", "PN", "P1"),
    ("GSM5284975", "PP", "P2"), ("GSM5284976", "PN", "P2"),
    ("GSM5284977", "PP", "P3"), ("GSM5284978", "PN", "P3"),
    ("GSM5284979", "PP", "P4"), ("GSM5284980", "PN", "P4"),
    ("GSM5284981", "PP", "P5"), ("GSM5284982", "PN", "P5"),
    ("GSM5284983", "PP", "P6"), ("GSM5284984", "PN", "P6"),
]

base = "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5284nnn"

for gsm, cond, patient in SAMPLES:
    gsm_dir = os.path.join(RAW_DIR, gsm)
    # Check if already downloaded
    if os.path.isdir(gsm_dir) and len(os.listdir(gsm_dir)) >= 3:
        print("  Already present:", gsm, cond)
        continue
    os.makedirs(gsm_dir, exist_ok=True)
    print("  Downloading", gsm, cond, "...", end=" ", flush=True)
    # Try downloading supplementary tar.gz or individual files
    tar_url = f"{base}/{gsm}/suppl/{gsm}_RAW.tar.gz"
    tar_path = os.path.join(RAW_DIR, gsm + "_RAW.tar.gz")
    try:
        urllib.request.urlretrieve(tar_url, tar_path)
        with tarfile.open(tar_path) as tf:
            tf.extractall(gsm_dir)
        os.remove(tar_path)
        print("OK (tar.gz)")
    except Exception:
        # Try individual files
        ok = True
        for fname in ["barcodes.tsv.gz", "features.tsv.gz", "matrix.mtx.gz"]:
            url = f"{base}/{gsm}/suppl/{gsm}_{fname}"
            dest = os.path.join(gsm_dir, fname)
            if os.path.exists(dest):
                continue
            try:
                urllib.request.urlretrieve(url, dest)
            except Exception as e:
                print("FAILED:", e)
                ok = False
                break
        if ok:
            print("OK (individual files)")

# Verify
found = sum(1 for gsm, _, _ in SAMPLES if os.path.isdir(os.path.join(RAW_DIR, gsm)))
print(f"Done. {found}/{len(SAMPLES)} sample directories ready.")


## Step 3: Script 01 — Load, QC, Condition Labelling

In [ ]:
%run ../src/01_load_qc.py


## Step 4: Script 02 — scVI Embedding + Leiden Clustering (GPU)

In [ ]:
%run ../src/02_scvi_embed.py


## Step 5: Script 03 — Cell Type Annotation

In [ ]:
%run ../src/03_annotate_clusters.py


## Step 6: Script 04 — Psoriasis Signature Scoring

In [ ]:
%run ../src/04_signature_scoring.py


## Step 7: Script 05 — Differential Expression (PP vs PN)

In [ ]:
%run ../src/05_differential_expression.py


## Step 8: Script 06 — LINCS L1000 Reversal Scoring

In [ ]:
%run ../src/06_lincs_repurposing.py


## Step 9: Script 07 — Novelty Assessment + Priority Scoring

In [ ]:
%run ../src/07_novelty_prioritization.py


## Step 10: Save outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil, os, glob
drive.mount("/content/drive")
DRIVE_OUTPUT = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
outputs = (glob.glob("/content/psoriasis-scrna/data/processed/*.csv") +
           glob.glob("/content/psoriasis-scrna/data/processed/*.h5ad"))
for f in outputs:
    shutil.copy2(f, os.path.join(DRIVE_OUTPUT, os.path.basename(f)))
    print("Saved:", os.path.basename(f))
print("All outputs saved to", DRIVE_OUTPUT)
